## Feature Enginerring

In [6]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../data/processed/weather_daily.csv") 

#### Ensure datetime and sort chronologically

In [11]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

In [13]:
df.head(5)

,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,2016-01-01,18.9,5.9,11.1,0.0,5.9
1,2016-01-02,18.4,4.2,10.4,0.0,8.0
2,2016-01-03,19.8,4.8,11.5,0.0,7.6
3,2016-01-04,17.0,9.4,12.7,3.7,10.9
4,2016-01-05,18.3,7.4,12.2,0.0,10.2


#### Create target variable: tomorrow's max temperature

In [14]:
df['tomorrow_max'] = df['temperature_2m_max'].shift(-1)

In [15]:
df[['date', 'temperature_2m_max', 'tomorrow_max']].tail(3)

,date,temperature_2m_max,tomorrow_max
3924,2026-09-29,29.6,30.6
3925,2026-09-30,30.6,31.1
3926,2026-10-01,31.1,NaN


In [17]:
df['temperature_2m_max'].shape, df['tomorrow_max'].shape

((3927,), (3927,))

### Create lag features for maximum temperature

In [18]:
df['temp_max_lag1'] = df['temperature_2m_max'].shift(1)
df['temp_max_lag2'] = df['temperature_2m_max'].shift(2)
df['temp_max_lag7'] = df['temperature_2m_max'].shift(7)

### Feature Engineering: Lag Features (Autoregulation & Momentum)

- **Why lag features?** Daily weather has strong autocorrelation (temporal memory); today's weather depends heavily on past atmospheric conditions.
- **`temp_max_lag1` (Yesterday's Max)**: Captures immediate persistence.
- **`temp_max_lag2` (2 Days Ago Max)**: Helps detect short-term trends/momentum (e.g., warming or cooling patterns).
- **`temp_max_lag7` (Same Day Last Week)**: Captures weekly patterns and longer-term baseline shifts.
- **Note on initial NaNs**: A positive shift like `.shift(k)` looks $k$ days into the past. For the first $k$ rows, that history does not exist in the dataset, which naturally produces `NaN` values.

In [19]:
df[['date', 'temperature_2m_max', 'temp_max_lag1', 'temp_max_lag2', 'temp_max_lag7']].head(8)

,date,temperature_2m_max,temp_max_lag1,temp_max_lag2,temp_max_lag7
0,2016-01-01,18.9,NaN,NaN,NaN
1,2016-01-02,18.4,18.9,NaN,NaN
2,2016-01-03,19.8,18.4,18.9,NaN
3,2016-01-04,17.0,19.8,18.4,NaN
4,2016-01-05,18.3,17.0,19.8,NaN
5,2016-01-06,17.9,18.3,17.0,NaN
6,2016-01-07,18.8,17.9,18.3,NaN
7,2016-01-08,16.9,18.8,17.9,18.9


#### Rolling 7-day average: Tells you the overall baseline / climate state of the current week. Is the area going through a sustained heatwave or a cold spell?

A single day can have a random fluctuation (e.g., sudden storm), but a 7-day or 14-day rolling average smoothes out the noise to reveal the underlying trend.

In [23]:
df['temp_max_roll_mean_7'] = df['temperature_2m_max'].rolling(window=7).mean()
df['temp_max_roll_std_7'] = df['temperature_2m_max'].rolling(window=7).std()

In [24]:
df[['date', 'temperature_2m_max', 'temp_max_roll_mean_7', 'temp_max_roll_std_7']].head(10)

,date,temperature_2m_max,temp_max_roll_mean_7,temp_max_roll_std_7
0,2016-01-01,18.9,NaN,NaN
1,2016-01-02,18.4,NaN,NaN
2,2016-01-03,19.8,NaN,NaN
3,2016-01-04,17.0,NaN,NaN
4,2016-01-05,18.3,NaN,NaN
5,2016-01-06,17.9,NaN,NaN
6,2016-01-07,18.8,18.442857,0.873417
7,2016-01-08,16.9,18.157143,1.014655
8,2016-01-09,17.2,17.985714,1.066815
9,2016-01-10,18.1,17.742857,0.723089


#### 1. What is a "Sliding Window"?
Imagine a rectangular frame that is 7 rows tall.

Pandas slides this 7-row frame down the dataframe row by row.

At each row, it looks at: the current row + the previous 6 rows (a total of 7 days).

#### Key Observations:
Index 0 to 5 (First 6 rows): They are NaN because Pandas requires all 7 days by default to compute the average.

Index 6 (7th day): This is the first non-NaN value because indices 0, 1, 2, 3, 4, 5, 6 give exactly 7 historical days.

std() (Standard Deviation): Measures how much temperatures fluctuated during that 7-day period.

In [25]:
# Calendar features
df['month'] = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear

df[['date', 'month', 'day_of_year']].head(3)

C:\Users\abdur\AppData\Local\Temp\ipykernel_5828\2006574279.py:3: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df['day_of_year'] = df['date'].dt.dayofyear


,date,month,day_of_year
0,2016-01-01,1,1
1,2016-01-02,1,2
2,2016-01-03,1,3


#### Advanced Seasonality — Cyclical Encoding (Sine & Cosine)Now, as your supervisor, here is a critical ML lesson:
The Problem with Raw Numbers for CyclesIn numbers, day_of_year = 365 (Dec 31) and day_of_year = 1 (Jan 1) look very far apart (distance = 364).   In reality, they are consecutive days in the middle of winter! A linear model will treat them as opposite ends of a spectrum instead of neighbor days.   The Solution: Trigonometric Transformation (Sin & Cos)We project the 365-day calendar onto a 2D circleOn a circle, day 365 and day 1 end up right next to each other!

In [26]:
# Cyclical encoding of day of year
df['sin_day_of_year'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
df['cos_day_of_year'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

df[['date', 'day_of_year', 'sin_day_of_year', 'cos_day_of_year']].head(3)

,date,day_of_year,sin_day_of_year,cos_day_of_year
0,2016-01-01,1,0.017202,0.999852
1,2016-01-02,2,0.034398,0.999408
2,2016-01-03,3,0.051584,0.998669


#### The "Why" & Possibilities for Missing Values
Look at where missing values are located:

The first 7 rows (indices 0 to 6) have NaN values due to temp_max_lag7 (looking 7 days back) and the 7-day rolling window.

The very last row (index 3926) has NaN in tomorrow_max because there is no recorded day after the last date.

#### Possibilities:
Imputation (filling with mean/median): Bad idea for time-series lags and targets, because making up past or target future temperatures introduces synthetic data noise.

Dropping rows with NaNs: Dropping 8 rows out of 3,927 rows (~0.2% of data) leaves us with a completely clean, real-world dataset.

In [27]:
df_clean = df.dropna().reset_index(drop=True)

In [29]:
print("Original data shape",df.shape)
print("Cleaned data shape",df_clean.shape)

Original data shape (3927, 16)
Cleaned data shape (3919, 16)


In [28]:
# Save the final engineered dataset
df_clean.to_csv("../data/processed/weather_features.csv", index=False)